Reference:https://towardsdatascience.com/augmenting-llms-with-rag-f79de914e672

In [ ]:
import langchain
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.embeddings.cache import CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

In [ ]:
from dotenv import load_dotenv
 
load_dotenv()

RAG Setup

We need the following to implement RAG with LangChain:

* Storage: A local store for our data we are providing for our RAG application. To scale up you can utilize other stores such as S3 as this gets larger
* Embeddings model: To create embedding out of the provided data, we use OpenAI Embeddings
* Vector Store: Store model embeddings, FAISS in this case
* Chain: Stitches together these different components, our LLM models is OpenAI in this case

In [ ]:
# where our embeddings will be stored
store = LocalFileStore("./cache")

In [ ]:
# instantiate a loader. If there are multiple pdf, and we have to load them together then use PyPDFDirectoryLoader
loader = PyPDFLoader("Enqurious_ETL_DB document.pdf")

In [ ]:
pages = loader.load_and_split()

In [ ]:
print(len(pages))

In [ ]:
# instantiate the embeddings model
embedding_model = OpenAIEmbeddings()

In [ ]:
# pass in our vector store
embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding_model,
    store
)

Reference: https://python.langchain.com/docs/modules/data_connection/text_embedding/caching_embeddings

CacheBackedEmbeddings will store the embeddings in our Local storage in a folder called ./cache via object that we created above: ```store = LocalFileStore("./cache")```. We can give other name also.

It is good to store it on cloud

FAISS REFERNCE: https://ai.meta.com/tools/faiss/#:~:text=FAISS%20(Facebook%20AI%20Similarity%20Search,more%20scalable%20similarity%20search%20functions.

In [ ]:
# !pip install faiss-cpu

In [ ]:
# pass in our vector store
vector_store = FAISS.from_documents(
    pages,
    embedder
)

In [ ]:
etl_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(),
    # chain_type="stuff", # by default chain type is stuff only, so no need to mention it explicitly
    retriever=vector_store.as_retriever(),
    return_source_documents=True,
    verbose=True
)

In [ ]:
import openai

In [ ]:
prompt = "How many tables are there in ETL database?"

In [ ]:
#vanilla OpenAI Response, without RAG
response = openai.Completion.create(
    engine="text-davinci-003",
    prompt=prompt,
    max_tokens=500
)

In [ ]:
print(response["choices"][0]["text"])

As it is observed that before embedding, the model doesn't have any definitive answer for our question.

In [ ]:
# The response after using embeddings
response_rag = etl_chain({"query": prompt})

In [ ]:
response_rag

In [ ]:
# Give clear instructions to find the output
prompt2 = "Refer the ETL Data Dictionary and find how many tables are there in ETL database?"

> Now let's ask this question to model with no context

In [ ]:
response = openai.Completion.create(
    engine="text-davinci-003",
    prompt=prompt2,
    max_tokens=500
)

In [ ]:
print(response["choices"][0]["text"])

> Now provide the same question to model with context

In [ ]:
# The response after using embeddings
response_rag = etl_chain({"query": prompt2})

In [ ]:
response_rag['result']

> We can observe that the model having context have successfully fetched the correct number of tables from the document

In [ ]:
prompt3 = "Refer the ETL Data Dictionary and return the names of tables present in ETL database?"

In [ ]:
response_rag = etl_chain({"query": prompt3})

In [ ]:
print(response_rag['result'])

In [ ]:
print(etl_chain({"query": 'What is the primary key of clients table?'})['result'])

In [ ]:
print(etl_chain({"query": 'What is the primary key of progress_fact table?'})['result'])

In [ ]:
# Now ask the model about the granularity of the progress_fact table
response_rag = etl_chain({"query": 'What is the granularity level of the progress_fact table?'})
response_rag['result']

In [ ]:
# Now ask the model about the granularity of the progress_fact table
response_rag = etl_chain({"query": 'What is the granularity level of the skills_fact table?'})
response_rag['result']

The responses given by the embedded model is correct to an extent.

In [ ]:
etl_chain({"query": 'Refer the data dictionary of Enqurious ETL document and frame the sql query for my question based on that. So how can we find top 5 learners of the Excel Essentials based on scores from the database?'})

In [ ]:
# Have copied the output from above and printed here
print("To find the top 5 learners of the Excel Essentials based on scores from the database, you can use the following SQL query:\n\n```sql\nSELECT learners_fact.learner_fact_id, learners_fact.order_id, learners_fact.client_id, learners_fact.score\nFROM learners_fact\nJOIN skills_fact ON learners_fact.skill_fact_id = skills_fact.skill_fact_id\nWHERE skills_fact.skill_name = 'Excel Essentials'\nORDER BY learners_fact.score DESC\nLIMIT 5;\n```\n\nThis query selects the necessary columns from the `learners_fact` table and joins it with the `skills_fact` table using the `skill_fact_id` column. It then filters the records based on the skill name 'Excel Essentials'. The results are sorted in descending order by the learner's score and limited to the top 5 learners.")

The above query is incorrect

In [ ]:
response_rag =  etl_chain({"query": "Which attribute represents the scores of the learners in skills_fact table?"})
response_rag

In [ ]:
print(response_rag['result'])

> Based on he above conversation it can be concluded that the theoretical questions are properly answered by the embedded model, but the sql query are not properly fetched out from that.